In [19]:
%%writefile AsymmetricMACrossStrategy.go
package strategy

import "math"

// Candle represents OHLCV data (chỉ cần Close)
type Candle struct {
	Close float64
}

// Signal output
type Signal struct {
	LongEntry  bool
	LongExit   bool
	ShortEntry bool
	ShortExit  bool
}

// Strategy struct
type AsymmetricMACrossStrategy struct {
	MA9   int
	MA50  int
	MA100 int
}

// Constructor
func NewAsymmetricMACrossStrategy(ma9, ma50, ma100 int) *AsymmetricMACrossStrategy {
	return &AsymmetricMACrossStrategy{
		MA9:   ma9,
		MA50:  ma50,
		MA100: ma100,
	}
}

// --- SMA ---
func calculateSMA(data []float64, period int) []float64 {
	sma := make([]float64, len(data))
	var sum float64

	for i := 0; i < len(data); i++ {
		sum += data[i]

		if i >= period {
			sum -= data[i-period]
		}

		if i >= period-1 {
			sma[i] = sum / float64(period)
		} else {
			sma[i] = math.NaN()
		}
	}

	return sma
}

// --- Apply Strategy ---
func (s *AsymmetricMACrossStrategy) Apply(candles []Candle) []Signal {
	n := len(candles)

	closePrices := make([]float64, n)
	for i := range candles {
		closePrices[i] = candles[i].Close
	}

	ma9 := calculateSMA(closePrices, s.MA9)
	ma50 := calculateSMA(closePrices, s.MA50)
	ma100 := calculateSMA(closePrices, s.MA100)

	signals := make([]Signal, n)

	for i := 1; i < n; i++ {
		// --- Entry: MA50 cross ABOVE MA100 ---
		entry := ma50[i] > ma100[i] &&
			ma50[i-1] <= ma100[i-1]

		// --- Exit: MA9 cross BELOW MA50 ---
		exit := ma9[i] < ma50[i] &&
			ma9[i-1] >= ma50[i-1]

		signals[i] = Signal{
			LongEntry:  entry,
			LongExit:   exit,
			ShortEntry: false,
			ShortExit:  false,
		}
	}

	return signals
}

Writing AsymmetricMACrossStrategy.go


In [11]:
import time
start_time = time.time()

In [12]:
!pip install --upgrade binance-historical-data plotly -q

In [13]:
import os
import warnings
import pandas as pd
pd.set_option('display.max_rows', None)
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime, timedelta, date
from binance_historical_data import BinanceDataDumper
warnings.filterwarnings('ignore')

In [14]:
def create_data_dumper(market_type="spot", data_frequency="1h"):
    return BinanceDataDumper(
        path_dir_where_to_dump=".",
        asset_class=market_type,
        data_type="klines",
        data_frequency=data_frequency,
    )

def get_list_all_trading_pairs(market_type="spot"):
    data_dumper = create_data_dumper(market_type)
    return data_dumper.get_list_all_trading_pairs()

def find_first_data_date(ticker, market_type="spot"):
    data_dumper = create_data_dumper(market_type)
    return data_dumper.get_min_start_date_for_ticker(ticker)

def filter_usdt_tickers(tickers):
    exclude_keywords = ["UPUSDT", "DOWNUSDT", "BEARUSDT", "BULLUSDT"]
    return [ticker for ticker in tickers if ticker.endswith("USDT") and not any(ex in ticker for ex in exclude_keywords)]

def find_first_data_date(ticker):
    data_dumper = BinanceDataDumper(
        path_dir_where_to_dump=".",
        asset_class="spot",
        data_type="klines",
        data_frequency="1h",
    )
    return data_dumper.get_min_start_date_for_ticker(ticker)

def format_time(seconds):
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    secs = int(seconds % 60)
    return f"{hours:02d}:{minutes:02d}:{secs:02d}"

def detect_timestamp_unit(timestamp):
    num_digits = len(str(timestamp))
    if num_digits == 13:
        return 'ms'
    elif num_digits == 16:
        return 'us'
    else:
        raise ValueError(f"Timestamp không hợp lệ: {timestamp}")

def convert_timestamp(timestamp):
    unit = detect_timestamp_unit(timestamp)
    return pd.to_datetime(timestamp, unit=unit, errors='coerce')

def download_ticker(ticker, date_start, date_end, data_frequency="1h", market_type="spot"):
    data_dumper = create_data_dumper(market_type, data_frequency)

    date_start = datetime.strptime(date_start, "%Y-%m-%d").date()
    date_end = datetime.strptime(date_end, "%Y-%m-%d").date()

    data_dumper.dump_data(
        tickers=ticker,
        date_start=date_start,
        date_end=date_end,
        is_to_update_existing=False,
    )

def read_csv_file(file_path):
    df = pd.read_csv(file_path)
    df.columns = [
        "open_time", "Open", "High", "Low", "Close", "volume",
        "close_time", "quote_asset_volume", "number_of_trades",
        "taker_buy_base_asset_volume", "taker_buy_quote_asset_volume", "ignore"
    ]
    df['open_time'] = df['open_time'].apply(convert_timestamp)
    df['close_time'] = df['close_time'].apply(convert_timestamp)
    return df

def get_csv_files(directory):
    try:
        if not os.path.exists(directory):
            print(f"Warning: Thư mục không tồn tại: {directory}")
            return []
        return [os.path.join(directory, f) for f in os.listdir(directory) if f.endswith('.csv')]
    except Exception as e:
        print(f"Lỗi khi đọc thư mục {directory}: {str(e)}")
        return []

# Load data from file
def process_csv_files(ticker, data_frequency="1h", market_type="spot"):
    base_path = f"{market_type}"

    daily_path = os.path.join(os.getcwd(), f"{base_path}/daily/klines/{ticker}/{data_frequency}")
    monthly_path = os.path.join(os.getcwd(), f"{base_path}/monthly/klines/{ticker}/{data_frequency}")

    daily_files = get_csv_files(daily_path)
    monthly_files = get_csv_files(monthly_path)

    all_files = daily_files + monthly_files

    if not all_files:
        print(f"❗ Không có file CSV nào cho {ticker} ({market_type})")
        return None

    data = pd.concat([read_csv_file(file) for file in all_files], ignore_index=True)
    data.sort_values(by='open_time', inplace=True)

    return data

In [15]:
class Backtester:
    def __init__(self, df, initial_balance=10000, risk_pct=1.0):
        self.df = df.reset_index(drop=True)
        self.initial_balance = initial_balance
        self.balance = initial_balance

        self.position = None
        self.entry_price = 0
        self.qty = 0

        self.fee_rate = 0.001
        self.slippage = 0.0005      # FIX: thêm slippage
        self.risk_pct = risk_pct

        self.trades = []
        self.equity_curve = []

        self.entry_fee = 0
        self.entry_time = None

    def run(self):
        for i in range(1, len(self.df)):
            t = self.df.iloc[i]["open_time"]
            bar = self.df.iloc[i]
            prev_bar = self.df.iloc[i-1]

            raw_price = bar["Open"]

            # ==========================================
            # STOP-AND-REVERSE
            # ==========================================

            # ===== LONG =====
            if self.position == "LONG":
                if prev_bar["long_exit"]:
                    exit_price = raw_price * (1 - self.slippage)
                    self.close_long(exit_price, t)

                    if prev_bar["short_entry"]:
                        entry_price = raw_price * (1 - self.slippage)
                        self.open_short(entry_price, t)

            # ===== SHORT =====
            elif self.position == "SHORT":
                if prev_bar["short_exit"]:
                    exit_price = raw_price * (1 + self.slippage)
                    self.close_short(exit_price, t)

                    if prev_bar["long_entry"]:
                        entry_price = raw_price * (1 + self.slippage)
                        self.open_long(entry_price, t)

            # ===== FLAT =====
            elif self.position is None:
                if prev_bar["long_entry"]:
                    entry_price = raw_price * (1 + self.slippage)
                    self.open_long(entry_price, t)
                elif prev_bar["short_entry"]:
                    entry_price = raw_price * (1 - self.slippage)
                    self.open_short(entry_price, t)

            # ===== EQUITY =====
            self.update_equity(t, bar["Close"])

    # ==========================================
    # EXECUTION
    # ==========================================

    def open_long(self, price, t):
        risk_amount = self.balance * self.risk_pct
        self.qty = risk_amount / price

        self.entry_fee = price * self.qty * self.fee_rate
        self.balance -= self.entry_fee

        self.entry_price = price
        self.position = "LONG"
        self.entry_time = t

    def close_long(self, price, t):
        pnl = (price - self.entry_price) * self.qty
        exit_fee = price * self.qty * self.fee_rate

        total_fee = self.entry_fee + exit_fee
        net_pnl = pnl - total_fee

        # FIX: accounting đúng
        self.balance += pnl - exit_fee

        self.record_trade(t, "LONG", self.entry_price, price, net_pnl)

        self.position = None
        self.qty = 0
        self.entry_price = 0

    def open_short(self, price, t):
        risk_amount = self.balance * self.risk_pct
        self.qty = risk_amount / price

        self.entry_fee = price * self.qty * self.fee_rate
        self.balance -= self.entry_fee

        self.entry_price = price
        self.position = "SHORT"
        self.entry_time = t

    def close_short(self, price, t):
        pnl = (self.entry_price - price) * self.qty
        exit_fee = price * self.qty * self.fee_rate

        total_fee = self.entry_fee + exit_fee
        net_pnl = pnl - total_fee

        self.balance += pnl - exit_fee

        self.record_trade(t, "SHORT", self.entry_price, price, net_pnl)

        self.position = None
        self.qty = 0
        self.entry_price = 0

    def record_trade(self, t, side, entry_p, exit_p, net_pnl):
        self.trades.append({
            "Entry_time": self.entry_time,
            "Exit_time": t,
            "Side": side,
            "Entry": entry_p,
            "Exit": exit_p,
            "PnL_net": net_pnl
        })

    def update_equity(self, t, close_price):
        equity = self.balance

        if self.position == "LONG":
            unrealized = (close_price - self.entry_price) * self.qty
            exit_fee = close_price * self.qty * self.fee_rate
            equity += unrealized - exit_fee

        elif self.position == "SHORT":
            unrealized = (self.entry_price - close_price) * self.qty
            exit_fee = close_price * self.qty * self.fee_rate
            equity += unrealized - exit_fee

        self.equity_curve.append({"time": t, "equity": equity})

def get_periods_per_year(tf: str):
    tf_map = {
        "1m": 365 * 24 * 60,
        "3m": 365 * 24 * 20,
        "5m": 365 * 24 * 12,
        "15m": 365 * 24 * 4,
        "30m": 365 * 24 * 2,
        "1h": 365 * 24,
        "2h": 365 * 12,
        "4h": 365 * 6,
        "6h": 365 * 4,
        "12h": 365 * 2,
        "1d": 365,
    }
    return tf_map.get(tf, 365)

def calculate_metrics(trades_df, equity_data, initial_balance, timeframe="1h"):
    eq_df = pd.DataFrame(equity_data).set_index("time")

    final_equity = eq_df["equity"].iloc[-1]
    total_return = (final_equity - initial_balance) / initial_balance

    running_max = eq_df["equity"].cummax()
    dd = (eq_df["equity"] - running_max) / running_max
    max_dd = dd.min()

    eq_df["returns"] = eq_df["equity"].pct_change().fillna(0)

    periods = get_periods_per_year(timeframe)

    std = eq_df["returns"].std()

    if std < 1e-8:
        sharpe = 0
    else:
        sharpe = eq_df["returns"].mean() / std * np.sqrt(periods)

    pnl_col = "PnL_net"

    total_trades = len(trades_df)
    win_rate = (trades_df[pnl_col] > 0).mean()

    gross_profit = trades_df[trades_df[pnl_col] > 0][pnl_col].sum()
    gross_loss = abs(trades_df[trades_df[pnl_col] < 0][pnl_col].sum())

    profit_factor = gross_profit / (gross_loss + 1e-9)

    return {
        "Final Equity": final_equity,
        "Total Return": total_return,
        "Max Drawdown": max_dd,
        "Sharpe": sharpe,
        "Trades": total_trades,
        "Winrate": win_rate,
        "Profit Factor": profit_factor,
        "equity_df": eq_df
    }

def resample_ohlc(df, timeframe):

    df = df.copy()
    df["open_time"] = pd.to_datetime(df["open_time"])
    df = df.set_index("open_time")

    ohlc_dict = {
        "Open": "first",
        "High": "max",
        "Low": "min",
        "Close": "last",
        "volume": "sum"
    }

    df_resampled = df.resample(timeframe).agg(ohlc_dict).dropna()

    df_resampled = df_resampled.reset_index()

    return df_resampled

TF_MAP = {
    "5m": "5T",
    "15m": "15T",
    "30m": "30T",
    "1h": "1H",
    "2h": "2H",
    "4h": "4H",
    "12h": "12H"
}

# ==========================================
# COMPARTMENT 3: ORCHESTRATOR (QUẢN LÝ DỰ ÁN)
# ==========================================
def run_multi_asset(tickers, from_date, to_date, timeframes, risk_pct=0.01):
    results = []

    # Khởi tạo chiến lược MỚI
    strategy = AsymmetricMACrossStrategy()

    for ticker in tickers:
        print(f"📥 BẮT ĐẦU XỬ LÝ: {ticker}")

        try:
            # 1. Download & Process Base Data
            download_ticker(ticker, from_date, to_date, "5m", market_type="spot")
            df_base = process_csv_files(ticker, "5m")

            if df_base is None or df_base.empty:
                print(f"❌ Dữ liệu rỗng cho {ticker}. Bỏ qua!")
                continue

            # Dọn dẹp cột thừa
            cols_to_drop = ['close_time','quote_asset_volume','number_of_trades','taker_buy_base_asset_volume','taker_buy_quote_asset_volume','ignore']
            df_base = df_base.drop(columns=[c for c in cols_to_drop if c in df_base.columns])
            df_base = df_base.sort_values("open_time").drop_duplicates("open_time")

        except Exception as e:
            print(f"❌ Lỗi tải dữ liệu {ticker}: {e}")
            continue

        for tf in timeframes:
            try:
                # 2. Resample Data
                df_tf = df_base.copy() if tf == "5m" else resample_ohlc(df_base, TF_MAP[tf])
                if len(df_tf) < 300:
                    print(f"   ⚠️ Lướt qua {tf:4}: Không đủ Data ({len(df_tf)} nến)")
                    continue

                # 3. Alpha Layer (Áp dụng chiến lược)
                df_signals = strategy.apply(df_tf)

                # 4. Execution Engine (Chạy Backtest)
                # Đã sửa lỗi gọi SimpleBacktester thành Backtester
                bt = Backtester(df_signals, risk_pct=risk_pct) # Khởi tạo với df
                bt.run()

                trades_df = pd.DataFrame(bt.trades)
                if trades_df.empty:
                    print(f"   ⚠️ Lướt qua {tf:4}: Bot móm (0 trades)")
                    continue

                # 5. Metrics & REAL-TIME TELEMETRY
                metrics = calculate_metrics(trades_df, bt.equity_curve, bt.initial_balance, timeframe=tf)
                metrics.update({"ticker": ticker, "timeframe": tf, "config": f"{ticker}_{tf}"})

                # Căn lề in log chuyên nghiệp
                ret_str = f"{metrics['Total Return']:>7.2%}"
                shp_str = f"{metrics['Sharpe']:>5.2f}"
                mdd_str = f"{metrics['Max Drawdown']:>7.2%}"
                trd_str = f"{metrics['Trades']:>4}"

                print(f"   ✅ Done {tf:4} | Total Return: {ret_str} | Sharpe: {shp_str} | MDD: {mdd_str} | Trades: {trd_str}")

                results.append({
                    "metrics": metrics,
                    "df_signals": df_signals,
                    "trades_df": trades_df
                })

            except Exception as e:
                print(f"   ❌ Lỗi ở {ticker}-{tf}: {e}")
                continue

    # ==========================================
    # TỔNG KẾT & VẼ BIỂU ĐỒ TOP 3
    # ==========================================
    if not results:
        print("\n❌ KHÔNG CÓ CẤU HÌNH NÀO RA LÃI/CÓ LỆNH.")
        return None

    results_df = pd.DataFrame([r["metrics"] for r in results])

    # Công thức tính điểm: Ưu tiên Sharpe cao, Lãi dương, Phạt nặng MDD cao
    results_df["score"] = (
        results_df["Trades"] *
        results_df["Sharpe"] *
        np.maximum(results_df["Total Return"], 0) /
        (results_df["Max Drawdown"].abs() + 1e-9)
    )

    results_df = results_df.sort_values("score", ascending=False).reset_index(drop=True)
    top3 = results_df.head(3)

    print("\n🏆 TOP 3 CẤU HÌNH XUẤT SẮC NHẤT:")
    print(top3[["ticker", "timeframe", "Total Return", "Sharpe", "Max Drawdown", "Trades"]])

    for _, row in top3.iterrows():
        config_data = next(r for r in results if r["metrics"]["config"] == row["config"])

        Visualizer.plot_equity(config_data["metrics"]["equity_df"], f"{row['config']} - Equity Curve")
        Visualizer.plot_ohlc_with_trades(config_data["df_signals"], config_data["trades_df"], f"{row['config']} - Trades History")

    return results_df

In [16]:
class AsymmetricMACrossStrategy:
    """
    Strategy: Asymmetric Trend-Following Crossover.
    Direction: LONG ONLY.
    Entry: MA50 crosses ABOVE MA100 (Medium-term Trend Confirmation).
    Exit: MA9 crosses BELOW MA50 (Short-term Weakness / Fast Exit).
    No Hard Stop Loss.
    """
    def __init__(self, ma9=9, ma50=50, ma100=100):
        self.ma9 = ma9
        self.ma50 = ma50
        self.ma100 = ma100

    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()

        # 1. Tính toán các đường Moving Average
        df["MA9"] = df["Close"].rolling(window=self.ma9).mean()
        df["MA50"] = df["Close"].rolling(window=self.ma50).mean()
        df["MA100"] = df["Close"].rolling(window=self.ma100).mean()

        # 2. Logic Vào lệnh (Entry)
        # MA50 cắt LÊN MA100: MA50 hiện tại > MA100 VÀ MA50 nến trước <= MA100 nến trước
        entry_condition = (df["MA50"] > df["MA100"]) & (df["MA50"].shift(1) <= df["MA100"].shift(1))

        # 3. Logic Thoát lệnh (Exit)
        # MA9 cắt XUỐNG MA50: MA9 hiện tại < MA50 VÀ MA9 nến trước >= MA50 nến trước
        exit_condition = (df["MA9"] < df["MA50"]) & (df["MA9"].shift(1) >= df["MA50"].shift(1))

        # 4. Gắn vào chuẩn 4 nút bấm API
        df["long_entry"] = entry_condition
        df["long_exit"] = exit_condition

        # Chiến lược này KHÔNG đánh Short
        df["short_entry"] = False
        df["short_exit"] = False

        return df

# ==========================================
# COMPARTMENT 2: PRESENTATION LAYER
# ==========================================
class Visualizer:
    @staticmethod
    def plot_equity(equity_df: pd.DataFrame, title: str):
        fig = go.Figure()
        fig.add_trace(go.Scatter(x=equity_df.index, y=equity_df["equity"], mode="lines", name="Equity", line=dict(color='gold')))
        fig.update_layout(title=title, template="plotly_dark", height=400)
        fig.show()

    @staticmethod
    def plot_ohlc_with_trades(df: pd.DataFrame, trades_df: pd.DataFrame, title: str):
        fig = make_subplots(
            rows=2, cols=1,
            shared_xaxes=True,
            row_heights=[0.7, 0.3],
            vertical_spacing=0.05
        )

        # ==========================================
        # 1. PRICE + MA
        # ==========================================
        fig.add_trace(go.Candlestick(
            x=df["open_time"],
            open=df["Open"],
            high=df["High"],
            low=df["Low"],
            close=df["Close"],
            name="OHLC"
        ), row=1, col=1)

        for col in ["MA9", "MA50", "MA100"]:
            if col in df.columns:
                fig.add_trace(go.Scatter(
                    x=df["open_time"],
                    y=df[col],
                    mode="lines",
                    name=col,
                    line=dict(width=1.5)
                ), row=1, col=1)

        # ==========================================
        # 2. TRADE MARKERS
        # ==========================================
        if not trades_df.empty:
            # Entry
            fig.add_trace(go.Scatter(
                x=trades_df["Entry_time"],
                y=trades_df["Entry"],
                mode="markers",
                marker=dict(
                    symbol="triangle-up",
                    size=12,
                    color="lime",
                    line=dict(width=1, color="black")
                ),
                name="Long Entry"
            ), row=1, col=1)

            # Exit
            fig.add_trace(go.Scatter(
                x=trades_df["Exit_time"],
                y=trades_df["Exit"],
                mode="markers",
                marker=dict(
                    symbol="triangle-down",
                    size=5,
                    color="red"
                ),
                name="Exit"
            ), row=1, col=1)

        # ==========================================
        # 3. RSI SUBPLOT
        # ==========================================
        if "RSI" in df.columns:
            fig.add_trace(go.Scatter(
                x=df["open_time"],
                y=df["RSI"],
                mode="lines",
                name="RSI"
            ), row=2, col=1)

            # Overbought / Oversold lines
            fig.add_hline(y=70, line_dash="dash", line_color="red", row=2, col=1)
            fig.add_hline(y=30, line_dash="dash", line_color="green", row=2, col=1)

        # ==========================================
        # 4. LAYOUT
        # ==========================================
        fig.update_layout(
            title=title,
            template="plotly_dark",
            height=800,
            xaxis_rangeslider_visible=False
        )

        fig.show()

In [17]:
tickers = ['BTCUSDT','ETHUSDT','XRPUSDT','BNBUSDT','SOLUSDT', 'TRXUSDT','DOGEUSDT','ADAUSDT','BCHUSDT','LINKUSDT']
# tickers = ['XRPUSDT']

timeframes = ["5m", "15m", "30m", "1h", "2h", "4h", "12h"]
# timeframes = ["30m", "1h", "2h", "4h", "12h"]

results = run_multi_asset(
    tickers,
    "2023-01-01",
    "2025-01-01",
    timeframes,
    risk_pct=1
)

results

Output hidden; open in https://colab.research.google.com to view.

In [18]:
end_time = time.time()
elapsed_seconds = end_time - start_time

# Chuyển sang định dạng hh:mm:ss
hours, rem = divmod(elapsed_seconds, 3600)
minutes, seconds = divmod(rem, 60)

print(f"Thời gian xử lý: {int(hours):02d}:{int(minutes):02d}:{seconds:05.2f}")

Thời gian xử lý: 00:20:51.74
